# 0. Sobre — Censo Escolar

**Objetivo:** realizar um diagnóstico inicial de estrutura, qualidade, cobertura e distribuição dos dados, antes da integração das bases do Tech Challenge 3.

**Dicionário de dados:** preencher com as definições das variáveis relevantes após inspecionar a base.

**Perguntas norteadoras:** a estrutura está consistente? Há problemas de qualidade? A cobertura é adequada? Há valores atípicos que exigem investigação?

# 1. Configuração e importação de bibliotecas

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

# 2. Carga e inspeção dos dados

Verificar dimensões, tipos de colunas e primeiros registros.

In [3]:
caminho = Path("../data/raw/external/censo_escolar/escola.parquet")

df = pd.read_parquet(caminho)

print(f"Linhas: {len(df):,} | Colunas: {df.shape[1]}")
display(df.head())
df.info()

Linhas: 657,819 | Colunas: 57


,ano,sigla_uf,id_municipio,id_escola,rede,tipo_categoria_escola_privada,tipo_localizacao,tipo_localizacao_diferenciada,tipo_situacao_funcionamento,agua_potavel,...,etapa_ensino_fundamental_anos_iniciais,quantidade_matricula_infantil_pre_escola,quantidade_matricula_fundamental_anos_iniciais,quantidade_matricula_fundamental_2_ano,quantidade_matricula_zona_residencia_urbana,quantidade_matricula_zona_residencia_rural,quantidade_matricula_utiliza_transporte_publico,quantidade_docente_fundamental_anos_iniciais,quantidade_turma_fundamental_anos_iniciais,quantidade_turma_fundamental_anos_iniciais_integral
0,2024,SE,2802106,28025105,4,None,1,None,2,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2024,SE,2804706,28013581,3,None,2,None,2,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2024,SE,2805000,28004671,3,None,1,None,2,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,2024,SE,2806701,28021282,3,None,2,None,2,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2024,SE,2800308,28034473,4,None,1,None,2,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 657819 entries, 0 to 657818
Data columns (total 57 columns):
 #   Column                                               Non-Null Count   Dtype 
---  ------                                               --------------   ----- 
 0   ano                                                  657819 non-null  Int64 
 1   sigla_uf                                             657819 non-null  object
 2   id_municipio                                         657819 non-null  object
 3   id_escola                                            657819 non-null  object
 4   rede                                                 657819 non-null  object
 5   tipo_categoria_escola_privada                        127293 non-null  object
 6   tipo_localizacao                                     657819 non-null  object
 7   tipo_localizacao_diferenciada                        541368 non-null  object
 8   tipo_situacao_funcionamento                          657819 non-

# 3. Qualidade dos dados

Consolidar valores ausentes e duplicatas em uma única verificação. Diferenciar duplicata exata de repetição de chave de negócio.

In [ ]:
qualidade = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "nulos_%": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(dropna=True)
}).sort_values("nulos_%", ascending=False)

display(qualidade)
print("Linhas duplicadas exatas:", df.duplicated().sum())

## 3.1 Chaves e regras de negócio

Validar somente as chaves que representem efetivamente a granularidade da base. **A chave abaixo é uma hipótese de trabalho e deve ser confirmada com o dicionário de dados.**

In [ ]:
chave_proposta = []
if chave_proposta and all(c in df.columns for c in chave_proposta):
    print("Nulos na chave:", df[chave_proposta].isna().any(axis=1).sum())
    print("Repetições da chave proposta:", df.duplicated(chave_proposta).sum())
else:
    print("Chave não definida ou colunas ausentes: revisar granularidade antes do diagnóstico.")

In [ ]:
numericas = df.select_dtypes(include="number").columns
if len(numericas):
    display(df[numericas].describe().T.round(2))
else:
    print("Nenhuma coluna numérica identificada.")

### Observações de qualidade

- **Inconsistências identificadas:** preencher após executar.
- **Regras adicionais:** aplicar conforme definição oficial dos indicadores; não tratar automaticamente zeros ou negativos como erros.

# 4. Análise exploratória dos dados

Verificar cobertura e visualizar somente os indicadores úteis para esta base.

In [ ]:
dimensoes = ['ano', 'id_municipio']
for coluna in dimensoes:
    if coluna in df.columns:
        valores = df[coluna].dropna()
        print(f"{coluna}: {valores.nunique():,} valores distintos")
        if coluna == "ano":
            display(df[coluna].value_counts(dropna=False).sort_index().rename("registros"))
        else:
            display(df[coluna].value_counts(dropna=False).head(10).rename("registros"))

## 4.1 Distribuição de indicadores educacionais

Selecionar um indicador representativo, evitando múltiplos gráficos com a mesma informação.

In [ ]:
indicador = ''  # substituir se a coluna tiver outro nome
if indicador and indicador in df.columns and pd.api.types.is_numeric_dtype(df[indicador]):
    display(df[indicador].describe().round(2))
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x=indicador, bins=25)
    plt.title(f"Distribuição — {indicador}")
    plt.tight_layout()
    plt.show()
else:
    print("Defina um indicador numérico existente para visualizar sua distribuição.")

## 4.2 Validações específicas

Investigar, quando aplicável, percentuais fora da escala, códigos inválidos, cobertura por ano, denominadores ausentes, valores incompatíveis e outliers relevantes. **Evitar regras genéricas não justificadas.**

In [ ]:
# Exemplo de regra: habilite APENAS se o indicador for uma taxa na escala 0–100.
# coluna_taxa = "taxa_alfabetizacao"
# if coluna_taxa in df.columns:
#     inconsistentes = df[coluna_taxa].notna() & ~df[coluna_taxa].between(0, 100)
#     print("Taxas fora do intervalo [0, 100]:", inconsistentes.sum())

# 5. Conclusão geral

**Estrutura e cobertura:** preencher com os achados reais.

**Qualidade dos dados:** listar somente problemas efetivamente encontrados.

**Impacto para a integração:** registrar chaves, diferenças de anos, nível de agregação e ajustes necessários.

**Decisão:** pronta para integração / exige investigação / exige tratamento (definir após execução).